# Проект 1



## 1. Подключение

In [1]:
import itertools
import math

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC: так они у всех одинаковые
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

подключились как: kausmanova


In [2]:
# Общее оформление графиков (акцентный цвет только у того, о чем вывод, остальное серым)
ACCENT = "#C1272D"
GRAY = "#9CA3AF"
DARK = "#374151"
sns.set_theme(style="white", rc={
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#E5E7EB",
})
plt.rcParams["figure.dpi"] = 110

TARIFF = {1: "Базовый", 2: "Расширенный"}


def mln(x: float, digits: int = 2) -> str:
    """Рубли -> строка в млн руб. с запятой: 11325107 -> '11,33'."""
    return f"{x / 1e6:,.{digits}f}".replace(",", " ").replace(".", ",")


def pct(x: float, digits: int = 1) -> str:
    """Доля -> строка в процентах с запятой: 0.0403 -> '4,0 %'."""
    return f"{x * 100:.{digits}f}".replace(".", ",") + " %"

## 2. Разведочный анализ: как я выбрала вопрос

Сначала провела четыре разведочных запроса: справочник цен, выручка по месяцам и тарифам, приток и отток подписок, выручка по каналам продаж. Нужно посмотреть, где за весь датасет что-то заметно изменилось

In [3]:
# 2.1 Справочник цен: когда и как менялась абонентская плата
tariffs = q("""
    select tariff_id, tariff_name, monthly_fee, share_slots, valid_from, valid_to
    from prime.tariffs
    order by tariff_id, valid_from
""")
tariffs

,tariff_id,tariff_name,monthly_fee,share_slots,valid_from,valid_to
0,1,Базовый,199.0,3,2024-01-01,2026-02-28
1,1,Базовый,249.0,3,2026-03-01,2030-12-31
2,2,Расширенный,399.0,5,2024-01-01,2026-02-28
3,2,Расширенный,429.0,5,2026-02-01,2026-06-30
4,2,Расширенный,449.0,5,2026-03-01,2030-12-31


In [4]:
# 2.2 Выручка по месяцам и тарифам

# Одна строка = один месяц x один тариф

# Откуда данные:
#   prime.payments p       - платежи (p)
#   prime.subscriptions s  - подписки (s)
#   join ... using (subscription_id) соединяет платеж с его подпиской
#   У каждого платежа ровно одна подписка, поэтому строк после соединения столько же сколько платежей
#   subscription_members НЕ беру (двойной счет из-за шеринга, проверка в разделе 3)

# Что считаю в каждой колонке:
#   month       - месяц платежа: to_char превращает момент в строку месяца
#   tariff_id   - тариф: он есть только в подписке, поэтому и нужен join
#   paying_subs - сколько разных (!!) подписок заплатили за месяц (distinct это без повторов)
#   n_success   - сколько списаний прошло
#   n_failed    - сколько списаний не прошло
#   revenue     - выручка: сумма только успешных списаний

# filter (where ...) - посчитать только строки, где условие выполнено (получаю и успешные, и неуспешные списания)



revenue = q("""
    select to_char(p.paid_at, 'YYYY-MM')                                          as month,
           s.tariff_id,
           count(distinct p.subscription_id) filter (where p.status = 'success')  as paying_subs,
           count(*) filter (where p.status = 'success')                           as n_success,
           count(*) filter (where p.status = 'failed')                            as n_failed,
           sum(p.amount) filter (where p.status = 'success')                      as revenue
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    group by 1, 2
    order by 1, 2
""")
# group by 1, 2 - сгруппировать по 1-й и 2-й колонке (месяц и тариф)
# order by 1, 2 - отсортировать по месяцу, внутри месяца по тарифу


# Средний чек по тарифу: выручка / число успешных списаний
revenue["avg_check"] = (revenue["revenue"] / revenue["n_success"]).round(2)
revenue.pivot(index="month", columns="tariff_id", values=["revenue", "avg_check"])

revenue             avg_check       
tariff_id           1           2         1      2
month                                             
2025-01    10415262.0  12726105.0     199.0  399.0
2025-02    12150343.0  14820057.0     199.0  399.0
2025-03    14284220.0  17434704.0     199.0  399.0
2025-04    14565805.0  17865225.0     199.0  399.0
2025-05    15789257.0  19406562.0     199.0  399.0
2025-06    16025072.0  19608456.0     199.0  399.0
2025-07    17222853.0  21226002.0     199.0  399.0
2025-08    17934079.0  22010037.0     199.0  399.0
2025-09    18024823.0  22014027.0     199.0  399.0
2025-10    19291657.0  23570127.0     199.0  399.0
2025-11    19231957.0  23489928.0     199.0  399.0
2025-12    20504960.0  25073559.0     199.0  399.0
2026-01    21083453.0  25822482.0     199.0  399.0
2026-02    19549959.0  23949576.0     199.0  399.0
2026-03    27710961.0  30518081.0     249.0  449.0
2026-04    25558356.0  28228181.0     249.0  449.0
2026-05    28898691.0  31865081.0     249.0  449.0
2026-06    28575738.0  31580415.0     249.0  449.0
2026-07    30135723.0  33263267.0     249.0  449.0
2026-08    28272705.0  31039819.0     249.0  449.0

In [5]:
# 2.3 Приток и отток подписок по месяцам

# Одна строка результата = один месяц
# Число подписчиков меняется двумя способами (кто-то оформляет, другие отменяют)

# Запрос состоит из двух частей (with ... as):

# new_subs: сколько подписок оформили за месяц
#   month     - месяц оформления (started_at)
#   new_subs  - всего новых подписок
#   new_trial - из них оформлены в триальный период (is_trial = true)

# cancels: сколько подписок отменили за месяц
#   беру только подписки, у которых есть дата отмены (ended_at is not null)
#   month     - месяц отмены (ended_at)
#   cancelled - всего отмен
#   дальше отмены по причинам:
#     too_expensive  - дорого
#     not_using      - не пользуется
#     payment_failed - не прошла оплата
#     switched       - перешел на другой сервис
#     other          - другое

# В конце соединяю две части по месяцу (!!!)
# full join нужен, чтобы не потерять месяц если в нем были только оформления или только отмены
# Обычный join оставил бы только месяцы, где есть и то и другое
# Если в месяце не было отмен, в колонках отмен будет NaN

flows = q("""
    with new_subs as (
        select to_char(started_at, 'YYYY-MM') as month,
               count(*)                         as new_subs,
               count(*) filter (where is_trial) as new_trial
        from prime.subscriptions
        group by 1
    ),
    cancels as (
        select to_char(ended_at, 'YYYY-MM')                              as month,
               count(*)                                                  as cancelled,
               count(*) filter (where cancel_reason = 'too_expensive')   as too_expensive,
               count(*) filter (where cancel_reason = 'not_using')       as not_using,
               count(*) filter (where cancel_reason = 'payment_failed')  as payment_failed,
               count(*) filter (where cancel_reason = 'switched')        as switched,
               count(*) filter (where cancel_reason = 'other')           as other
        from prime.subscriptions
        where ended_at is not null
        group by 1
    )
    select *
    from new_subs
    full join cancels using (month)
    order by month
""")

flows.tail(12)

,month,new_subs,new_trial,cancelled,too_expensive,not_using,payment_failed,switched,other
20,2025-09,10620,1791,5712.0,1334.0,1973.0,834.0,996.0,575.0
21,2025-10,11376,1894,6052.0,1483.0,2014.0,882.0,1106.0,567.0
22,2025-11,10759,1908,6081.0,1448.0,2068.0,831.0,1101.0,633.0
23,2025-12,11438,1985,6354.0,1538.0,2205.0,847.0,1132.0,632.0
24,2026-01,11326,1898,6607.0,1644.0,2208.0,849.0,1199.0,707.0
25,2026-02,10154,1727,6246.0,1519.0,2167.0,859.0,1133.0,568.0
26,2026-03,11315,1963,6945.0,1677.0,2371.0,961.0,1247.0,689.0
27,2026-04,10746,1845,6880.0,1669.0,2298.0,973.0,1291.0,649.0
28,2026-05,11028,1835,7202.0,1679.0,2466.0,1013.0,1278.0,766.0
29,2026-06,11031,1869,7119.0,1785.0,2386.0,983.0,1261.0,704.0


In [6]:
# 2.4 Выручка по каналам продаж по месяцам

# Проверить не влияет ли какой-то один канал на общую выручку сильнее остальных

# Одна строка результата запроса = один месяц x один канал

# Откуда беру данные:
#   prime.payments p       - платежи
#   prime.subscriptions s  - подписки, в них есть номер канала partner_id
#   prime.partners pr      - справочник каналов, в нем есть название канала

# Соединения (строк после соединений столько же, сколько платежей, без двойного счета):
#   платеж в подписку по subscription_id (у платежа ровно одна подписка)
#   подписка в канал по partner_id (у подписки ровно один канал)

# Что считаю:
#   month        - месяц платежа
#   partner_name - название канала
#   paying_subs  - сколько разных подписок заплатили за месяц
#   revenue      - выручка, сумма только успешных списаний

by_partner = q("""
    select to_char(p.paid_at, 'YYYY-MM')                                          as month,
           pr.partner_name,
           count(distinct p.subscription_id) filter (where p.status = 'success')  as paying_subs,
           sum(p.amount) filter (where p.status = 'success')                      as revenue
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    join prime.partners pr using (partner_id)
    group by 1, 2
    order by 1, 2
""")

# Разворачиваю таблицу через (pivot):
# месяцы по строкам, каналы по столбцам, в ячейках выручка

by_partner.pivot(index="month", columns="partner_name", values="revenue").tail(8)

partner_name,Автодилер,Банк-партнёр,Маркетплейс,Мобильный оператор,Онлайн-кинотеатр,Прямые продажи,Ритейл-сеть
month,,,,,,,
2026-01,6727686.0,6711555.0,6722618.0,6724956.0,6703794.0,6706379.0,6608947.0
2026-02,6243846.0,6219707.0,6207504.0,6242364.0,6234304.0,6229895.0,6121915.0
2026-03,8365057.0,8342586.0,8374162.0,8333106.0,8320976.0,8297534.0,8195621.0
2026-04,7742499.0,7718159.0,7708895.0,7693344.0,7695336.0,7623067.0,7605237.0
2026-05,8706972.0,8724805.0,8712052.0,8718143.0,8680427.0,8631551.0,8589822.0
2026-06,8617215.0,8646699.0,8615396.0,8648065.0,8611447.0,8530626.0,8486705.0
2026-07,9082411.0,9120949.0,9074205.0,9079546.0,9118805.0,8984887.0,8938187.0
2026-08,8503983.0,8503119.0,8447636.0,8522250.0,8542647.0,8417909.0,8374980.0


### Что было найдено в разведке
